# 🎛️ Module 12: Hyperparameter Tuning
## Theory + Practical

---

## 📚 THEORY

### Parameters vs Hyperparameters
```
Parameters      → Learned by the model during training  (weights, biases)
Hyperparameters → Set by YOU before training  (C, max_depth, n_estimators)
```

### Tuning Strategies

| Method | How | Pros | Cons |
|--------|-----|------|------|
| **GridSearchCV** | Try ALL combinations | Exhaustive | Slow with many params |
| **RandomizedSearchCV** | Try random sample | Faster | May miss best combo |
| **Bayesian Optimization** | Smart search using past results | Efficient | Complex |

### Rule of Thumb
- Start with **RandomizedSearchCV** to find the ballpark
- Then use **GridSearchCV** around the best region

---

## 🔬 PRACTICAL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print('Ready ✅')

### Practical 1: GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 5, 10],
    'min_samples_split': [2, 5]
}  # 3 × 3 × 2 = 18 combinations × 5 folds = 90 model fits

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X, y)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Score:   {grid_search.best_score_:.4f}')

### Practical 2: RandomizedSearchCV (Faster)

In [ ]:
from scipy.stats import randint

# Wider search space — sample randomly instead of exhaustive
param_dist = {
    'n_estimators':    randint(50, 500),
    'max_depth':       [None, 3, 5, 10, 15, 20],
    'min_samples_split': randint(2, 20),
    'max_features':    ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=30,       # Only try 30 random combinations
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X, y)

print(f'Best Parameters: {random_search.best_params_}')
print(f'Best CV Score:   {random_search.best_score_:.4f}')

print(f'\nGrid Search:   exhaustive search = 90 fits')
print(f'Random Search: 30 random fits — much faster, similar results!')

### Practical 3: Visualize Search Results

In [ ]:
import pandas as pd

results_df = pd.DataFrame(random_search.cv_results_)
results_df = results_df.sort_values('rank_test_score')

plt.figure(figsize=(10, 4))
plt.scatter(range(len(results_df)), results_df['mean_test_score'],
            c=results_df['rank_test_score'], cmap='RdYlGn', s=60)
plt.axhline(random_search.best_score_, color='red', linestyle='--',
            label=f'Best: {random_search.best_score_:.4f}')
plt.title('RandomizedSearchCV — All 30 Trials')
plt.xlabel('Trial (sorted by rank)')
plt.ylabel('CV Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print('Top 5 configurations:')
print(results_df[['param_n_estimators','param_max_depth','mean_test_score']].head().to_string())

### 🏋️ Try It Yourself!

In [ ]:
# ✏️ YOUR TURN: Tune Gradient Boosting hyperparameters
gb_params = {
    'n_estimators':  [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth':     [2, 3, 5]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params, cv=5, scoring='accuracy', n_jobs=-1
)
gb_grid.fit(X, y)

print(f'Best GB Params: {gb_grid.best_params_}')
print(f'Best CV Score:  {gb_grid.best_score_:.4f}')

## 🎓 Summary

| Concept | Takeaway |
|---------|----------|
| Hyperparameters | Set by you, not learned |
| GridSearchCV | Exhaustive, use for small grids |
| RandomizedSearchCV | Faster, use for large grids |
| n_jobs=-1 | Use all CPU cores — speeds up search |
| cv=5 | Always use cross-validation to evaluate |

➡️ **Next: End-to-End ML Project (Capstone)**